In [ ]:
import cv2
import numpy as np
import os
import mediapipe as mp
from tqdm import tqdm

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

class ASLDataCollector:
    def __init__(self):
        self.DATA_PATH = 'MP_Data'
        # YOUR EXACT CLASSES from CollectData.py
        self.actions = np.array([
            'A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
            '0','1','2','3','4','5','6','7','8','9',
            ' ','THANKYOU','HELLO','HI','YES','NO'  # SPACE + YOUR WORDS
        ])
        self.no_sequences = 30
        self.sequence_length = 30
        
        os.makedirs(self.DATA_PATH, exist_ok=True)
        self.model = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.7)
    
    def mediapipe_detection(self, image):
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        results = self.model.process(image_rgb)
        image_rgb.flags.writeable = True
        image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        return image, results
    
    def draw_landmarks(self, image, results):
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style())
        return image
    
    def extract_keypoints(self, results):
        rh = np.zeros(21*3)
        lh = np.zeros(21*3)
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                if hand_landmarks.landmark[0].x < 0.5:
                    lh = np.array([[lmk.x, lmk.y, lmk.z] for lmk in hand_landmarks.landmark]).flatten()
                else:
                    rh = np.array([[lmk.x, lmk.y, lmk.z] for lmk in hand_landmarks.landmark]).flatten()
        return np.concatenate([lh, rh])
    
    def collect_data(self):
        cap = cv2.VideoCapture(0)
        for action in self.actions:
            os.makedirs(os.path.join(self.DATA_PATH, action), exist_ok=True)
        
        print(f"Collecting {self.no_sequences} sequences per class")
        print("Press SPACE to START recording, Q=quit")
        
        sequence = []
        for action_idx, action in enumerate(tqdm(self.actions, desc="Classes")):
            print(f"\n📁 ACTION: '{action}'")
            
            for seq_idx in range(self.no_sequences):
                print(f"  Seq {seq_idx+1}/{self.no_sequences} - Show '{action}' → SPACE")
                sequence = []
                
                while True:
                    ret, frame = cap.read()
                    if not ret: break
                    
                    image, results = self.mediapipe_detection(frame)
                    image = self.draw_landmarks(image, results)
                    keypoints = self.extract_keypoints(results)
                    sequence.append(keypoints)
                    
                    cv2.putText(image, f'{len(sequence)}/{self.sequence_length}', (10, 50),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
                    cv2.putText(image, action.upper(), (10, 100),
                               cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,0), 3)
                    
                    cv2.imshow('ASL Keypoints Collection', image)
                    key = cv2.waitKey(1) & 0xFF
                    
                    if len(sequence) == self.sequence_length:
                        break
                    if key == ord('q'):
                        cap.release()
                        cv2.destroyAllWindows()
                        return
                
                # Save
                np.save(os.path.join(self.DATA_PATH, action, f'seq_{seq_idx}'), np.array(sequence))
                print(f"  ✅ Saved MP_Data/{action}/seq_{seq_idx}.npy")
        
        cap.release()
        cv2.destroyAllWindows()
        print("🎉 ALL DATA COLLECTED!")

if __name__ == "__main__":
    collector = ASLDataCollector()
    collector.collect_data()